# Experiment: Alpha-Globin Rebalancing Gate

Objective: decide whether direct alpha-globin reduction / chain rebalancing should become a Nakafa Lab therapy lane, assay benchmark, or rejected idea.

Success criteria:
- recover source-backed human modifier, RNA/gene-therapy, and editing evidence;
- separate advanced-therapy evidence from affordable first-lab candidates;
- return a clear gate decision without patient treatment advice.


In [ ]:
from __future__ import annotations

import json
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Any

ROOT = Path.cwd()
PUBMED_DIR = ROOT / "data" / "literature" / "pubmed"
SELECTED_XML = PUBMED_DIR / "2026-04-27-alpha-globin-rebalancing-selected-abstracts.xml"

SNAPSHOT_PATHS = {
    "therapeutic_reduction": PUBMED_DIR
    / "2026-04-27-alpha-globin-reduction-therapeutic-search.json",
    "sirna_reduction": PUBMED_DIR
    / "2026-04-27-alpha-globin-sirna-beta-thalassemia-search.json",
    "crispr_hba_locus": PUBMED_DIR
    / "2026-04-27-crispr-alpha-globin-locus-beta-thalassemia-search.json",
    "copy_number_modifier": PUBMED_DIR
    / "2026-04-27-alpha-globin-triplication-beta-thalassemia-search.json",
    "alpha_thal_coinheritance": PUBMED_DIR
    / "2026-04-27-alpha-thalassemia-coinheritance-beta-modifier-search.json",
    "mcsr2_query_control": PUBMED_DIR
    / "2026-04-27-mcsr2-alpha-globin-beta-thalassemia-search.json",
}

EVIDENCE_GROUPS = {
    "human_genetic_modifier": {"7509151", "18026953", "3012927", "38895064"},
    "mouse_or_cell_rna_reduction": {"18556409", "17716993", "17493845"},
    "integrated_gene_therapy_vector": {"33940155"},
    "ex_vivo_hsc_or_locus_editing": {"33635334", "30911135"},
    "small_molecule_probe_only": {"27810991"},
}

REQUIRED_READOUTS = [
    "HBA1/HBA2 or alpha-globin output",
    "alpha/non-alpha globin balance",
    "HbH-like over-suppression safety boundary",
    "free or insoluble alpha-globin",
    "HBG1/HBG2, HbF protein, and F-cell percentage",
    "erythroid maturation",
    "viability, apoptosis, and mature red-cell hemolysis",
]

## Load Source Snapshots

The snapshot files are the reproducible evidence inputs. The zero-result MCS-R2 query is kept as a useful query-control, not as proof that the enhancer biology is absent.


In [ ]:
def load_snapshot(path: Path) -> dict[str, Any]:
    """Read one compact PubMed JSON snapshot."""
    return json.loads(path.read_text())


def snapshot_summary(name: str, snapshot: dict[str, Any]) -> dict[str, object]:
    """Return a compact count-and-query row for one PubMed snapshot."""
    results = snapshot.get("results")
    if not isinstance(results, list):
        results = []

    return {
        "lane": name,
        "query": snapshot.get("query"),
        "count": snapshot.get("count"),
        "top_pmids": [str(item.get("pmid")) for item in results[:3]],
    }


snapshots = {name: load_snapshot(path) for name, path in SNAPSHOT_PATHS.items()}

snapshot_rows = [
    snapshot_summary(name, snapshot) for name, snapshot in snapshots.items()
]
snapshot_rows

## Extract Selected Abstract Metadata

This parses the selected PubMed XML bundle so the gate can reason over PMIDs, titles, journals, years, and abstracts without manual copying.


In [ ]:
def compact_text(element: ET.Element | None) -> str:
    """Collapse XML text content into one readable string."""
    if element is None:
        return ""

    return " ".join("".join(element.itertext()).split())


def parse_selected_articles(path: Path) -> dict[str, dict[str, str]]:
    """Parse selected PubMed XML into PMID-keyed article metadata."""
    root = ET.parse(path).getroot()
    articles: dict[str, dict[str, str]] = {}

    for item in root.findall(".//PubmedArticle"):
        pmid = item.findtext(".//PMID") or ""
        if not pmid:
            continue

        title = compact_text(item.find(".//ArticleTitle"))
        abstract_parts = [
            compact_text(part) for part in item.findall(".//AbstractText")
        ]

        articles[pmid] = {
            "title": title,
            "journal": item.findtext(".//Journal/Title") or "",
            "year": item.findtext(".//JournalIssue/PubDate/Year") or "",
            "abstract_head": " ".join(abstract_parts)[:260],
        }

    return articles


selected_articles = parse_selected_articles(SELECTED_XML)
selected_article_count = len(selected_articles)
selected_article_count

## Evidence Gate

The lane should be promoted only if it has modifier genetics plus intervention proof-of-concept. It stays out of the first affordable lab panel if delivery or safety makes it an advanced-therapy benchmark.


In [ ]:
def group_evidence_rows(
    groups: dict[str, set[str]],
    articles: dict[str, dict[str, str]],
) -> list[dict[str, object]]:
    """Summarize which evidence groups are covered by selected PMIDs."""
    rows: list[dict[str, object]] = []

    for group, pmids in groups.items():
        covered_pmids = sorted(pmid for pmid in pmids if pmid in articles)
        rows.append(
            {
                "group": group,
                "covered_pmids": covered_pmids,
                "covered_count": len(covered_pmids),
                "example_titles": [
                    articles[pmid]["title"] for pmid in covered_pmids[:2]
                ],
            }
        )

    return rows


def gate_decision(rows: list[dict[str, object]]) -> dict[str, object]:
    """Convert evidence coverage into a conservative research decision."""
    covered_groups = {
        str(row["group"])
        for row in rows
        if isinstance(row.get("covered_count"), int) and row["covered_count"] > 0
    }

    core_groups = {
        "human_genetic_modifier",
        "mouse_or_cell_rna_reduction",
        "integrated_gene_therapy_vector",
        "ex_vivo_hsc_or_locus_editing",
    }
    has_core_support = core_groups.issubset(covered_groups)

    if not has_core_support:
        decision = "hold_for_more_source_retrieval"
    else:
        decision = "advanced_chain_balance_benchmark_not_first_panel_lead"

    return {
        "lane": "alpha_globin_rebalancing",
        "decision": decision,
        "covered_groups": sorted(covered_groups),
        "missing_core_groups": sorted(core_groups - covered_groups),
        "required_readouts": REQUIRED_READOUTS,
    }


evidence_rows = group_evidence_rows(EVIDENCE_GROUPS, selected_articles)
decision = gate_decision(evidence_rows)
{"evidence_rows": evidence_rows, "decision": decision}

## Interpretation

Alpha-globin reduction is mechanistically strong because beta-thalassemia damage is driven by alpha/non-alpha globin-chain imbalance. The retrieved evidence supports it as a root-cause-adjacent benchmark, but not as a cheap immediate compound lane.

Key boundary: too much alpha-globin suppression could create alpha-thalassemia / HbH-like risk. Any experiment must measure balance and red-cell safety, not just lower alpha-globin.


In [ ]:
final_result = {
    "decision": decision["decision"],
    "why_it_matters": [
        "human alpha-globin copy-number biology modifies beta-thalassemia severity",
        "RNAi and antisense approaches improved model phenotypes",
        "HSC or vector strategies can rebalance globin chains in advanced settings",
    ],
    "why_not_first_affordable_lead": [
        "delivery is currently RNA, vector, HSC, or editing heavy",
        "over-suppression can create alpha-thalassemia or HbH-like risk",
        "IOX1 is an epigenetic probe signal, not a clean patient candidate",
    ],
    "next_repo_action": "document as chain-balance benchmark and assay gate",
}
final_result

## Decision

Promote alpha-globin rebalancing to a high-value chain-balance benchmark and assay gate.

Do not add it to the first affordable wet-lab quote panel as a direct treatment candidate. Use it to judge whether HbF, autophagy, pyruvate-kinase, or future candidates truly improve globin-chain balance without creating new red-cell harm.
